# CS180 Project 4B: Feature Matching for Autostitching

**Name:** Puthyrath Hy

Spec: https://cal-cs180.github.io/fa26/hw/proj3/partB.html · Paper: Brown, Szeliski & Winder,
*Multi-Image Matching using Multi-Scale Oriented Patches* (MOPS)

**How to run:** locally, `pip install -r requirements.txt`, then run all cells. On Colab, open https://colab.research.google.com/github/rath-hy/cs180-proj4/blob/main/partB.ipynb and run all cells (the first cell clones the repo). Uses the Part A photos in
`data/proj 4 images/` and the Part A correspondences in `points/` (for the manual vs. automatic comparison).
`harris.py` is the course's sample code. Results are saved to `output/` with a `b_` prefix.

**Conventions (same as Part A):** points are `(x, y) = (column, row)`; images are float64 in `[0, 1]`.
`harris.py` returns `(row, col)`, which is converted to `(x, y)` in `detect_corners`.

In [ ]:
# Colab only: clone the repo (photos, points, harris.py) and run from inside it. Does nothing locally.
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.path.exists('harris.py'):
    if not os.path.exists('cs180-proj4'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/rath-hy/cs180-proj4.git'], check=True)
    os.chdir('cs180-proj4')
print(os.getcwd())

## Setup

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.ndimage import gaussian_filter

from harris import get_harris_corners, dist2

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
np.set_printoptions(precision=4, suppress=True)

DATA, POINTS, OUT = Path('data'), Path('points'), Path('output')
OUT.mkdir(exist_ok=True)
MAX_DIM = 1600  # same downscaling as Part A, so the Part A points line up

### From Part A

Copied from `main.ipynb` unchanged, except `homographies_to_ref`, which now takes the pairwise
homographies as an argument so it works for both manual and automatic ones.

In [ ]:
def load_image(path, max_dim=MAX_DIM):
    """Load an image as float64 RGB in [0, 1], honoring EXIF rotation and downscaling."""
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    if max_dim and max(im.size) > max_dim:
        s = max_dim / max(im.size)
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    return np.asarray(im, dtype=np.float64) / 255.0


def save_image(im, name):
    """Save a float image in [0, 1] (RGB or grayscale) to output/<name>."""
    arr = (np.clip(im, 0, 1) * 255).round().astype(np.uint8)
    Image.fromarray(arr).save(OUT / name)
    return OUT / name


def show_images(images, titles=None, ncols=None, size=5, save=None, suptitle=None):
    """Display a list of images in a grid. `save` writes the whole figure to output/<save>."""
    n = len(images)
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for i, im in enumerate(images):
        ax = axes.flat[i]
        ax.imshow(np.clip(im, 0, 1), cmap='gray' if im.ndim == 2 else None)
        if titles:
            ax.set_title(titles[i])
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    if save:
        fig.savefig(OUT / save, bbox_inches='tight', dpi=150)
    plt.show()


def pair_name(name, i, j):
    return f'{name}_{i}_{j}'


def load_points(name, shapes=None):
    """Load points/<name>.json -> list of (n, 2) arrays, rescaled to `shapes` if given."""
    d = json.loads((POINTS / f'{name}.json').read_text())
    pts = [np.array(p, dtype=np.float64).reshape(-1, 2) for p in d['points']]
    if shapes is not None:
        for i, (old, new) in enumerate(zip(d['shapes'], shapes)):
            pts[i] = pts[i] * np.array([new[1] / old[1], new[0] / old[0]])
    return pts


def apply_homography(H, pts):
    """Map (n, 2) points through H, with the homogeneous divide."""
    p = np.c_[pts, np.ones(len(pts))] @ H.T
    return p[:, :2] / p[:, 2:3]


def reprojection_error(H, pts1, pts2):
    """Mean pixel distance between H(pts1) and pts2."""
    return np.linalg.norm(apply_homography(H, pts1) - pts2, axis=1).mean()


def timed(fn, *args, **kwargs):
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - t0


def build_homography_system(im1_pts, im2_pts):
    """Build A (2n x 8) and b (2n,) for the unknowns h = [h11, ..., h32], with h33 = 1."""
    A, b = [], []
    for (x, y), (xp, yp) in zip(im1_pts, im2_pts):
        A.append([x, y, 1, 0, 0, 0, -x * xp, -y * xp])
        b.append(xp)
        A.append([0, 0, 0, x, y, 1, -x * yp, -y * yp])
        b.append(yp)
    return np.array(A, dtype=np.float64), np.array(b, dtype=np.float64)


def computeH(im1_pts, im2_pts):
    """Least-squares homography mapping im1_pts -> im2_pts."""
    assert len(im1_pts) == len(im2_pts) >= 4, 'need at least 4 correspondences'
    A, b = build_homography_system(im1_pts, im2_pts)
    h, *_ = np.linalg.lstsq(A, b, rcond=None)
    return np.append(h, 1.0).reshape(3, 3)


def compute_warp_bounds(im_shape, H):
    """Bounding box (xmin, ymin, xmax, ymax) of the image's corners mapped through H."""
    h, w = im_shape[:2]
    corners = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float64)
    warped = apply_homography(H, corners)
    xmin, ymin = np.floor(warped.min(axis=0)).astype(int)
    xmax, ymax = np.ceil(warped.max(axis=0)).astype(int) + 1
    return int(xmin), int(ymin), int(xmax), int(ymax)


def _source_coords(im, H, bounds):
    """Map every output pixel back through H^-1; returns source (sx, sy) grids."""
    if bounds is None:
        bounds = compute_warp_bounds(im.shape, H)
    xmin, ymin, xmax, ymax = bounds
    xs, ys = np.meshgrid(np.arange(xmin, xmax), np.arange(ymin, ymax))
    src = apply_homography(np.linalg.inv(H), np.stack([xs.ravel(), ys.ravel()], axis=1))
    return src[:, 0].reshape(xs.shape), src[:, 1].reshape(xs.shape)


def warpImageBilinear(im, H, bounds=None):
    """Inverse warp with bilinear interpolation. Returns (imwarped, alpha)."""
    sx, sy = _source_coords(im, H, bounds)
    h, w = im.shape[:2]
    valid = (sx >= 0) & (sx <= w - 1) & (sy >= 0) & (sy <= h - 1)
    x, y = sx[valid], sy[valid]
    x0 = np.minimum(np.floor(x).astype(int), w - 2)
    y0 = np.minimum(np.floor(y).astype(int), h - 2)
    shape = (-1,) + (1,) * (im.ndim - 2)
    a, b = (x - x0).reshape(shape), (y - y0).reshape(shape)
    out = np.zeros(sx.shape + im.shape[2:])
    out[valid] = ((1 - a) * (1 - b) * im[y0, x0] + a * (1 - b) * im[y0, x0 + 1]
                  + (1 - a) * b * im[y0 + 1, x0] + a * b * im[y0 + 1, x0 + 1])
    return out, valid.astype(np.float64)


def make_alpha_mask(h, w):
    """Feathering weights: 1 at the center, falling linearly to ~0 at the edges."""
    ramp_y = 1 - np.abs(np.linspace(-1, 1, h))
    ramp_x = 1 - np.abs(np.linspace(-1, 1, w))
    return np.minimum.outer(ramp_y, ramp_x) + 1e-6


def build_mosaic(images, Hs, warp_fn=None):
    """Warp every image into the reference frame (Hs[i]: image i -> reference) and feather-blend."""
    warp_fn = warp_fn or warpImageBilinear
    bs = np.array([compute_warp_bounds(im.shape, H) for im, H in zip(images, Hs)])
    bounds = (int(bs[:, 0].min()), int(bs[:, 1].min()), int(bs[:, 2].max()), int(bs[:, 3].max()))
    rgbs, wgts = [], []
    for im, H in zip(images, Hs):
        warped, _ = warp_fn(np.dstack([im, make_alpha_mask(*im.shape[:2])]), H, bounds)
        rgbs.append(warped[..., :3])
        wgts.append(warped[..., 3:])
    return sum(w * x for w, x in zip(wgts, rgbs)) / np.maximum(sum(wgts), 1e-8)


def homographies_to_ref(pair_H, name, n, ref):
    """Chain adjacent-pair homographies (pair_H[(name, i, i+1)]: i -> i+1) so Hs[i] maps image i to ref."""
    Hs = [None] * n
    Hs[ref] = np.eye(3)
    for i in range(ref - 1, -1, -1):
        Hs[i] = Hs[i + 1] @ pair_H[(name, i, i + 1)]
    for i in range(ref + 1, n):
        Hs[i] = Hs[i - 1] @ np.linalg.inv(pair_H[(name, i - 1, i)])
    return [H / H[2, 2] for H in Hs]

### Images

In [ ]:
P = DATA / 'proj 4 images'
MOSAICS = {
    'wall': dict(paths=[P / 'wall_1.JPG', P / 'wall_2.JPG', P / 'wall_3.JPG'], ref=1),
    'door': dict(paths=[P / 'door_a_1.JPG', P / 'door_a_2.JPG', P / 'door_a_3.JPG'], ref=1),
    'chimney': dict(paths=[P / 'chimney_1.JPG', P / 'chimney_2.JPG', P / 'chimney_3.JPG'], ref=1),
}
mosaic_images = {name: [load_image(p) for p in cfg['paths']] for name, cfg in MOSAICS.items()}


def to_gray(im):
    return im @ np.array([0.299, 0.587, 0.114])


gray_images = {name: [to_gray(im) for im in ims] for name, ims in mosaic_images.items()}
for name, ims in mosaic_images.items():
    print(name, [im.shape for im in ims])

# B.1: Harris Corner Detection

In [ ]:
def detect_corners(gray, edge_discard=20):
    """Harris corners as (n, 2) (x, y) points, plus their corner strengths (n,)."""
    h, coords = get_harris_corners(gray, edge_discard)
    rows, cols = coords
    return np.stack([cols, rows], axis=1).astype(np.float64), h[rows, cols]


def plot_points(ax, im, pts, title, color='r', size=3):
    ax.imshow(im)
    ax.plot(pts[:, 0], pts[:, 1], '.', color=color, ms=size)
    ax.set_title(title)
    ax.axis('off')

### Adaptive Non-Maximal Suppression

In [ ]:
def anms(pts, strengths, n_keep=500, c_robust=0.9, max_candidates=5000):
    """Keep the n_keep points with the largest suppression radius r_i =
    min ||x_i - x_j|| over j with f_i < c_robust * f_j."""
    order = np.argsort(-strengths)[:max_candidates]  # strongest candidates only, to bound memory
    pts, f = pts[order], strengths[order]
    D = dist2(pts, pts)
    D[~(f[:, None] < c_robust * f[None, :])] = np.inf
    r2 = D.min(axis=1)  # squared radius
    keep = np.argsort(-r2)[:n_keep]
    return pts[keep], f[keep], np.sqrt(r2[keep])

### Sanity check

In [ ]:
_pts = np.random.default_rng(0).uniform(0, 500, (300, 2))
_f = np.random.default_rng(1).random(300)
_kept, _, _r = anms(_pts, _f, n_keep=50)
assert len(_kept) == 50 and np.isinf(_r[0]) and np.allclose(_kept[0], _pts[_f.argmax()]), 'global max must survive with r = inf'
assert np.all(np.diff(_r[np.isfinite(_r)]) <= 0), 'radii should be sorted descending'
print('OK')

In [ ]:
N_KEEP = 500

features = {}  # name -> list of dicts, one per image
for name, grays in gray_images.items():
    features[name] = []
    for gray in grays:
        pts, f = detect_corners(gray)
        anms_pts, anms_f, _ = anms(pts, f, N_KEEP)
        features[name].append(dict(harris=pts, harris_f=f, pts=anms_pts, f=anms_f))
    print(name, 'Harris corners per image:', [len(d['harris']) for d in features[name]])

for name in MOSAICS:
    ref = MOSAICS[name]['ref']
    im, d = mosaic_images[name][ref], features[name][ref]
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    plot_points(axes[0], im, d['harris'], f'{name}[{ref}]: all Harris corners ({len(d["harris"])})', size=1)
    plot_points(axes[1], im, d['pts'], f'{name}[{ref}]: after ANMS ({len(d["pts"])})', size=5)
    plt.tight_layout()
    fig.savefig(OUT / f'b_{name}_corners.jpg', bbox_inches='tight', dpi=150)
    plt.show()

For comparison, the 500 *strongest* Harris corners (no ANMS), on the same image:

In [ ]:
for name in MOSAICS:
    ref = MOSAICS[name]['ref']
    im, d = mosaic_images[name][ref], features[name][ref]
    top = d['harris'][np.argsort(-d['harris_f'])[:N_KEEP]]
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    plot_points(axes[0], im, top, f'{name}[{ref}]: {N_KEEP} strongest corners', size=5)
    plot_points(axes[1], im, d['pts'], f'{name}[{ref}]: {N_KEEP} corners from ANMS', size=5)
    plt.tight_layout()
    fig.savefig(OUT / f'b_{name}_top_vs_anms.jpg', bbox_inches='tight', dpi=150)
    plt.show()

**B.1 write-up:**

_TODO_

# B.2: Feature Descriptor Extraction

In [ ]:
WINDOW, DESC_SIZE = 40, 8
SPACING = WINDOW // DESC_SIZE  # 5
OFFSETS = np.arange(DESC_SIZE) * SPACING - (WINDOW // 2 - SPACING // 2)  # [-18, -13, ..., 17], centered


def extract_descriptors(gray, pts, sigma=2.0):
    """Axis-aligned 8x8 descriptors sampled every 5 px from a blurred 40x40 window,
    bias/gain normalized. Returns (n, 64)."""
    blurred = gaussian_filter(gray, sigma)
    xs = pts[:, 0].astype(int)[:, None, None] + OFFSETS[None, None, :]
    ys = pts[:, 1].astype(int)[:, None, None] + OFFSETS[None, :, None]
    patches = blurred[ys, xs]  # (n, 8, 8)
    patches = patches - patches.mean(axis=(1, 2), keepdims=True)
    patches = patches / (patches.std(axis=(1, 2), keepdims=True) + 1e-8)
    return patches.reshape(len(pts), -1)


def raw_window(gray, pt):
    x, y = int(pt[0]), int(pt[1])
    return gray[y - WINDOW // 2:y + WINDOW // 2, x - WINDOW // 2:x + WINDOW // 2]

### Sanity check

In [ ]:
_g = gaussian_filter(np.random.default_rng(0).random((200, 200)), 2)
_p = np.array([[60., 70.], [120., 100.]])
_d = extract_descriptors(_g, _p)
assert _d.shape == (2, 64)
assert np.allclose(_d.mean(axis=1), 0) and np.allclose(_d.std(axis=1), 1, atol=1e-6)
assert np.allclose(extract_descriptors(3 * _g + 0.5, _p), _d, atol=1e-6), 'should be invariant to aI + b'
print('OK')

In [ ]:
for name, grays in gray_images.items():
    for gray, d in zip(grays, features[name]):
        d['desc'] = extract_descriptors(gray, d['pts'])

# Show a few descriptors from one image: location, raw 40x40 window, normalized 8x8 descriptor.
name, idx, n_show = 'wall', MOSAICS['wall']['ref'], 6
gray, d = gray_images[name][idx], features[name][idx]
picks = np.random.default_rng(0).choice(len(d['pts']), n_show, replace=False)

fig = plt.figure(figsize=(16, 9))
ax = fig.add_subplot(1, 3, 1)
ax.imshow(mosaic_images[name][idx])
for k, i in enumerate(picks):
    x, y = d['pts'][i]
    ax.add_patch(plt.Rectangle((x - WINDOW / 2, y - WINDOW / 2), WINDOW, WINDOW, fill=False, ec='yellow', lw=1.5))
    ax.text(x + 25, y, str(k), color='yellow', fontsize=12, weight='bold')
ax.set_title(f'{name}[{idx}]: sampled features'); ax.axis('off')
for k, i in enumerate(picks):
    a = fig.add_subplot(n_show, 6, 6 * k + 3)
    a.imshow(raw_window(gray, d['pts'][i]), cmap='gray'); a.axis('off')
    a.set_title(f'{k}: 40x40 window' if k == 0 else str(k), fontsize=9)
    a = fig.add_subplot(n_show, 6, 6 * k + 4)
    a.imshow(d['desc'][i].reshape(DESC_SIZE, DESC_SIZE), cmap='gray'); a.axis('off')
    if k == 0:
        a.set_title('8x8 descriptor', fontsize=9)
plt.tight_layout()
fig.savefig(OUT / 'b_descriptors.jpg', bbox_inches='tight', dpi=150)
plt.show()

**B.2 write-up:**

_TODO_

# B.3: Feature Matching

In [ ]:
RATIO = 0.5  # threshold on e_1NN / e_2NN (squared errors), from Figure 6b of the paper


def match_features(desc1, desc2, ratio=RATIO):
    """Lowe ratio test on squared SSD. Returns (idx1, idx2, ratios) for accepted matches."""
    E = dist2(desc1, desc2)
    nn = np.argsort(E, axis=1)[:, :2]
    e1 = E[np.arange(len(E)), nn[:, 0]]
    e2 = E[np.arange(len(E)), nn[:, 1]]
    r = e1 / np.maximum(e2, 1e-12)
    keep = r < ratio
    return np.flatnonzero(keep), nn[keep, 0], r[keep]


def plot_matches(im1, im2, p1, p2, title, inliers=None, save=None):
    """Draw matches as lines between side-by-side images. If `inliers` is given, outliers are red."""
    h = max(im1.shape[0], im2.shape[0])
    pad = lambda im: np.pad(im, ((0, h - im.shape[0]), (0, 0), (0, 0)))
    canvas, off = np.hstack([pad(im1), pad(im2)]), im1.shape[1]
    inliers = np.ones(len(p1), bool) if inliers is None else inliers
    fig, ax = plt.subplots(figsize=(18, 18 * h / canvas.shape[1]))
    ax.imshow(canvas)
    for (x1, y1), (x2, y2), ok in zip(p1, p2, inliers):
        c = 'lime' if ok else 'red'
        ax.plot([x1, x2 + off], [y1, y2], '-', color=c, lw=0.8, alpha=0.8)
        ax.plot([x1, x2 + off], [y1, y2], 'o', color=c, ms=3)
    ax.set_title(title); ax.axis('off')
    plt.tight_layout()
    if save:
        fig.savefig(OUT / save, bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
matches = {}  # (name, i, i+1) -> (pts_i, pts_j)
for name, ims in mosaic_images.items():
    F = features[name]
    for i in range(len(ims) - 1):
        a, b, r = match_features(F[i]['desc'], F[i + 1]['desc'])
        p1, p2 = F[i]['pts'][a], F[i + 1]['pts'][b]
        matches[(name, i, i + 1)] = (p1, p2)
        plot_matches(ims[i], ims[i + 1], p1, p2, f'{name} {i}->{i + 1}: {len(p1)} matches (ratio < {RATIO})',
                     save=f'b_{pair_name(name, i, i + 1)}_matches.jpg')

Distribution of the 1-NN / 2-NN ratio over all features (compare with Figure 6b):

In [ ]:
all_ratios = []
for name in MOSAICS:
    F = features[name]
    for i in range(len(F) - 1):
        E = np.sort(dist2(F[i]['desc'], F[i + 1]['desc']), axis=1)
        all_ratios.append(E[:, 0] / E[:, 1])
plt.figure(figsize=(8, 4))
plt.hist(np.concatenate(all_ratios), bins=50)
plt.axvline(RATIO, color='r', ls='--', label=f'threshold = {RATIO}')
plt.xlabel('$e_{1NN} / e_{2NN}$'); plt.ylabel('number of features'); plt.legend()
plt.savefig(OUT / 'b_ratio_hist.jpg', bbox_inches='tight', dpi=150)
plt.show()

**B.3 write-up:**

_TODO_

# B.4: RANSAC for Robust Homography

In [ ]:
def ransac_homography(p1, p2, n_iters=2000, eps=2.0, seed=0):
    """4-point RANSAC. Returns (H refit on all inliers, inlier mask)."""
    rng = np.random.default_rng(seed)
    best = np.zeros(len(p1), bool)
    for _ in range(n_iters):
        idx = rng.choice(len(p1), 4, replace=False)
        H = computeH(p1[idx], p2[idx])
        if not np.all(np.isfinite(H)):
            continue
        with np.errstate(divide='ignore', invalid='ignore'):
            err = np.linalg.norm(apply_homography(H, p1) - p2, axis=1)
        inliers = err < eps
        if inliers.sum() > best.sum():
            best = inliers
    return computeH(p1[best], p2[best]), best

### Sanity check

In [ ]:
_rng = np.random.default_rng(0)
_H = np.array([[1.1, 0.05, 30.0], [-0.03, 0.95, -12.0], [1e-4, -2e-4, 1.0]])
_p1 = _rng.uniform(0, 800, (100, 2))
_p2 = apply_homography(_H, _p1) + _rng.normal(0, 0.5, (100, 2))
_bad = _rng.random(100) < 0.4  # 40% outliers
_p2[_bad] = _rng.uniform(0, 800, (_bad.sum(), 2))
_Hr, _in = ransac_homography(_p1, _p2)
print(f'least squares on everything: {reprojection_error(computeH(_p1, _p2), _p1[~_bad], _p2[~_bad]):.1f} px; '
      f'RANSAC: {reprojection_error(_Hr, _p1[~_bad], _p2[~_bad]):.2f} px')
assert reprojection_error(_Hr, _p1[~_bad], _p2[~_bad]) < 1.5 and (_in == ~_bad).mean() > 0.95
print('OK')

In [ ]:
auto_H, manual_H = {}, {}
for (name, i, j), (p1, p2) in matches.items():
    ims = mosaic_images[name]
    H, inl = ransac_homography(p1, p2)
    auto_H[(name, i, j)] = H
    plot_matches(ims[i], ims[j], p1, p2, f'{name} {i}->{j}: {inl.sum()}/{len(p1)} RANSAC inliers (green)',
                 inliers=inl, save=f'b_{pair_name(name, i, j)}_ransac.jpg')

    # Part A manual correspondences: fit the manual H, and score both H's on the manual points
    m1, m2 = load_points(pair_name(name, i, j), [ims[i].shape, ims[j].shape])
    manual_H[(name, i, j)] = computeH(m1, m2)
    print(f'{name} {i}->{j}: error on manual points: manual H {reprojection_error(manual_H[(name, i, j)], m1, m2):.2f} px, '
          f'automatic H {reprojection_error(H, m1, m2):.2f} px')

### Mosaics: manual (Part A) vs. automatic

In [ ]:
auto_mosaics = {}
for name, ims in mosaic_images.items():
    n, ref = len(ims), MOSAICS[name]['ref']
    manual = build_mosaic(ims, homographies_to_ref(manual_H, name, n, ref))
    auto, secs = timed(build_mosaic, ims, homographies_to_ref(auto_H, name, n, ref))
    auto_mosaics[name] = auto
    show_images([manual, auto], [f'{name}: manual', f'{name}: automatic'], size=9,
                save=f'b_{name}_manual_vs_auto.jpg')
    save_image(manual, f'b_{name}_manual.jpg')
    save_image(auto, f'b_{name}_auto.jpg')

### Experiment: sloppier clicks

The Part A points were clicked carefully. To see how sensitive manual stitching is to click accuracy, this
**simulates** a less careful person: every manual point (in both images) is moved by Gaussian noise with
standard deviation `CLICK_SIGMA` pixels, and the homography is refit. The original Part A mosaic is unchanged.
Errors are measured on the original (careful) clicks.

In [ ]:
CLICK_SIGMA = 3.0  # px of simulated click noise, at MAX_DIM = 1600


def sloppy_H(name, i, j, sigma, seed=0):
    ims = mosaic_images[name]
    m1, m2 = load_points(pair_name(name, i, j), [ims[i].shape, ims[j].shape])
    rng = np.random.default_rng(seed)
    return computeH(m1 + rng.normal(0, sigma, m1.shape), m2 + rng.normal(0, sigma, m2.shape))


def error_on_clicks(H, name, i, j):
    ims = mosaic_images[name]
    m1, m2 = load_points(pair_name(name, i, j), [ims[i].shape, ims[j].shape])
    return reprojection_error(H, m1, m2)


# Error vs. click noise, averaged over pairs and 20 random draws
sigmas = [0, 1, 2, 3, 5, 8]
mean_err = [np.mean([error_on_clicks(sloppy_H(*k, s, seed), *k) for k in manual_H for seed in range(20)])
            for s in sigmas]
auto_err = np.mean([error_on_clicks(auto_H[k], *k) for k in manual_H])
plt.figure(figsize=(7, 4))
plt.plot(sigmas, mean_err, 'o-', label='manual H, refit on noisy clicks')
plt.axhline(auto_err, color='g', ls='--', label='automatic H (RANSAC)')
plt.xlabel('click noise $\\sigma$ (px)'); plt.ylabel('mean error on careful clicks (px)'); plt.legend()
plt.savefig(OUT / 'b_click_noise_curve.jpg', bbox_inches='tight', dpi=150)
plt.show()

sloppy_H_all = {k: sloppy_H(*k, CLICK_SIGMA) for k in manual_H}
for k in manual_H:
    print(f'{k[0]} {k[1]}->{k[2]}: careful {error_on_clicks(manual_H[k], *k):.2f} px, '
          f'sloppy (sigma={CLICK_SIGMA:g}) {error_on_clicks(sloppy_H_all[k], *k):.2f} px, '
          f'automatic {error_on_clicks(auto_H[k], *k):.2f} px')

In [ ]:
for name, ims in mosaic_images.items():
    n, ref = len(ims), MOSAICS[name]['ref']
    sloppy = build_mosaic(ims, homographies_to_ref(sloppy_H_all, name, n, ref))
    careful = np.asarray(Image.open(OUT / f'b_{name}_manual.jpg'), dtype=np.float64) / 255
    show_images([careful, sloppy, auto_mosaics[name]],
                [f'{name}: careful clicks', f'{name}: sloppy clicks (simulated, $\\sigma$={CLICK_SIGMA:g} px)',
                 f'{name}: automatic'], size=7, save=f'b_{name}_sloppy_vs_auto.jpg')
    save_image(sloppy, f'b_{name}_manual_sloppy.jpg')

Zoomed in on the left seam (same region of the reference frame in all three; misalignment shows up as ghosting):

In [ ]:
def mosaic_origin(ims, Hs):
    # Reference-frame (x, y) of the mosaic's top-left pixel, matching build_mosaic's canvas.
    bs = np.array([compute_warp_bounds(im.shape, H) for im, H in zip(ims, Hs)])
    return bs[:, 0].min(), bs[:, 1].min()


ZOOM_HALF = 150  # crop half-size in px
for name, ims in mosaic_images.items():
    n, ref = len(ims), MOSAICS[name]['ref']
    h = ims[ref].shape[0]
    cx, cy = 0, h // 2  # the left seam lies along x = 0 of the reference image
    crops, titles = [], []
    for label, pair_Hs in [('careful clicks', manual_H), (f'sloppy clicks ($\\sigma$={CLICK_SIGMA:g} px)', sloppy_H_all),
                           ('automatic', auto_H)]:
        Hs = homographies_to_ref(pair_Hs, name, n, ref)
        ox, oy = mosaic_origin(ims, Hs)
        x0, y0 = cx - ZOOM_HALF - ox, cy - ZOOM_HALF - oy
        crops.append(build_mosaic(ims, Hs)[y0:y0 + 2 * ZOOM_HALF, x0:x0 + 2 * ZOOM_HALF])
        titles.append(f'{name}: {label}')
    show_images(crops, titles, size=6, save=f'b_{name}_sloppy_zoom.jpg')

**B.4 write-up:**

_TODO_